# STEP 3A: Continuous Monitoring & Dashboards

## Real-Time Production Monitoring

This notebook demonstrates:
1. Real-time score distribution monitoring
2. PSI calculation and drift detection
3. Data quality tracking
4. Performance dashboards
5. Anomaly detection and reporting

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
from datetime import datetime
import json

print('[OK] Step 3A: Continuous Monitoring & Dashboards')
print(f'Date: {datetime.now().strftime("%Y-%m-%d")}')

## Test 1: Load Baseline Configuration

In [ ]:
print('='*70)
print('TEST 1: LOAD BASELINE CONFIGURATION')
print('='*70)

# Load baseline metrics from Step 1
baseline = {
    'composite_score': {
        'mean': 5.5,
        'std': 1.5,
        'median': 5.5,
        'gini': 0.20,
    },
    'psi_threshold': 0.25,
    'shift_alert_pct': 5.0,
}

print(f'\nBaseline Metrics Loaded:')
for metric, val in baseline['composite_score'].items():
    print(f'  {metric:10s}: {val:.2f}')
print(f'\nMonitoring Thresholds:')
print(f'  PSI threshold: {baseline["psi_threshold"]:.2f}')
print(f'  Shift alert: {baseline["shift_alert_pct"]:.1f}%')

## Test 2: Monitor Real-Time Scores

In [ ]:
print('\n' + '='*70)
print('TEST 2: MONITOR REAL-TIME SCORES')
print('='*70)

# Simulate streaming scores from production
np.random.seed(42)
current_scores = np.random.normal(5.5, 1.5, 1000).clip(1, 10)

current_metrics = {
    'mean': np.mean(current_scores),
    'std': np.std(current_scores),
    'median': np.median(current_scores),
    'gini': np.mean(current_scores),
}

print(f'\nCurrent Metrics (Last 1000 scores):')
for metric, val in current_metrics.items():
    baseline_val = baseline['composite_score'][metric]
    change_pct = ((val - baseline_val) / baseline_val) * 100
    status = '[OK]' if abs(change_pct) <= 5.0 else '[ALERT]'
    print(f'  {metric:10s}: {val:6.2f} (baseline: {baseline_val:5.2f}, change: {change_pct:+5.1f}%) {status}')

## Test 3: Calculate PSI Continuously

In [ ]:
print('\n' + '='*70)
print('TEST 3: CALCULATE PSI CONTINUOUSLY')
print('='*70)

def calculate_psi(baseline, current, n_bins=5):
    baseline_counts = np.histogram(baseline, bins=n_bins)[0] + 0.0001
    current_counts = np.histogram(current, bins=n_bins)[0] + 0.0001
    baseline_pct = baseline_counts / baseline_counts.sum()
    current_pct = current_counts / current_counts.sum()
    psi = (current_pct * np.log(current_pct / baseline_pct)).sum()
    return psi

# Simulate baseline from Step 1
baseline_scores = np.random.normal(5.5, 1.5, 500).clip(1, 10)
psi = calculate_psi(baseline_scores, current_scores)

print(f'\nPSI Analysis:')
print(f'  PSI Value: {psi:.4f}')
print(f'  Threshold: {baseline["psi_threshold"]:.4f}')
if psi < 0.10:
    print(f'  Status: [OK] No significant change')
elif psi < baseline['psi_threshold']:
    print(f'  Status: [WARNING] Small change')
else:
    print(f'  Status: [ALERT] Significant drift')

## Test 4: Generate Monitoring Dashboard

In [ ]:
print('\n' + '='*70)
print('TEST 4: GENERATE MONITORING DASHBOARD')
print('='*70)

dashboard = {
    'timestamp': datetime.now().isoformat(),
    'records_monitored': len(current_scores),
    'metrics': {
        'mean': float(current_metrics['mean']),
        'std': float(current_metrics['std']),
        'median': float(current_metrics['median']),
    },
    'psi': float(psi),
    'alerts': {
        'psi_alert': psi > baseline['psi_threshold'],
        'shift_alert': any(abs(((current_metrics[k] - baseline['composite_score'][k]) / baseline['composite_score'][k]) * 100) > baseline['shift_alert_pct'] for k in ['mean', 'median']),
    },
    'status': 'HEALTHY' if psi < baseline['psi_threshold'] else 'ALERT',
}

print(f'\nMonitoring Dashboard:')
print(json.dumps(dashboard, indent=2))

## STEP 3A Complete

In [ ]:
print('\n' + '='*70)
print('STEP 3A: CONTINUOUS MONITORING COMPLETE')
print('='*70)
print(f'\nMonitoring Summary:')
print(f'  Baseline loaded: OK')
print(f'  Real-time metrics: OK')
print(f'  PSI calculated: {psi:.4f}')
print(f'  Dashboard generated: OK')
print(f'\nStatus: {dashboard["status"]}')